# Hurricane Express — membership customer profiling**One export, one question: who are these members, and which of them are about to leave?**`hurricane_customer_profille_data.csv` is an event log for the membership book of two HurricaneExpress car-wash sites — 11,001 rows covering **466 customers / 587 vehicles** between**2025-09-13 and 2026-08-10**. Each row is one event (a `wash` or a `payment`) joined onto avehicle, so the file carries both sides of a subscription business: what members *pay* and whatthey *consume*.That combination is what makes the file worth modelling. A payments-only export tells you whensomeone left; a payments-**and**-usage export tells you *why*, early enough to act.### What this notebook does| § | Section | Output ||---|---------|--------|| 1 | Preprocessing — two traps in the file's shape | a clean event log || 2 | Data-quality audit | what to trust, what to drop || 3 | The book at a glance | growth, revenue, churn || 4 | Retention: cohorts and the renewal hazard | where members are actually lost || 5 | What drives churn | five tested hypotheses, one null || 6 | A churn model | AUC 0.71 holdout, calibrated, 2.7× top-decile lift || 7 | Segmentation → four personas | 16% of members = 35% of revenue || 8 | Unit economics and CLV | contribution per persona || 9 | Problem statements | what this data can and cannot answer |All shared logic lives in [`profiling.py`](profiling.py), which the Streamlit demo([`app.py`](app.py)) imports too — so the notebook's numbers and the demo's numbers cannot drift.```bashconda activate sonnysstreamlit run experiments/customer-profiling/app.py```

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from scipy import stats

sys.path.insert(0, str(Path.cwd()))
import profiling as P
import viz

pio.renderers.default = "notebook"
T = viz.theme(dark=False)          # the notebook renders on white
SEG_COLOR = viz.segment_colors(T)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

raw = pd.read_csv(P.DATA)
print(f"{len(raw):,} raw rows x {raw.shape[1]} columns")
raw.head(4)

---## 1. Preprocessing — two traps in the file's shapeThe export is one row per **(event × vehicle)**. That single fact decides almost every numberdownstream, and it cuts in opposite directions for the two event types.**Trap 1 — payments are fanned out across the household's vehicles.** A three-car household payingonce appears as three identical payment rows. Summing `amount` naively counts the charge threetimes.**Trap 2 — washes are *not* fanned out, and must not be collapsed.** A three-car household reallydoes drive three washes. The only duplicate wash rows are one `vehicle_id` carrying two spellingsof its plate.Getting these backwards — collapsing washes, or not collapsing payments — is the difference betweena correct P&L and a 58% revenue overstatement.

In [ ]:
# Trap 1: the same charge, repeated once per vehicle in the household.
c4 = raw[(raw.customer_id == 4) & (raw.event_type == "payment")]
c4[["customer_id", "vehicle_id", "vehicle_license", "event_date", "payment_type", "amount"]].head(6)

In [ ]:
# Trap 2: two rows, same vehicle_id, same second -- one wash, two spellings of the plate.
c86 = raw[(raw.customer_id == 86) & (raw.event_type == "wash") & (raw.event_date == "2025-12-07 18:23:20")]
c86[["customer_id", "vehicle_id", "vehicle_license", "vehicle_make", "event_date"]]

In [ ]:
ev = P.load_events()          # tidy log: parsed dates, "-" -> NaN, all-null columns dropped
pay = P.payments(ev)          # household-level charges (fan-out collapsed)
wsh = P.washes(ev)            # vehicle-washes (deduped on customer+vehicle+timestamp)

naive = raw[raw.event_type == "payment"].amount.sum()
pd.DataFrame({
    "rows in export": [(raw.event_type == "payment").sum(), (raw.event_type == "wash").sum()],
    "after handling": [len(pay), len(wsh)],
    "revenue naive":  [f"${naive:,.0f}", ""],
    "revenue correct": [f"${pay.amount.sum():,.0f}", ""],
}, index=["payments", "washes"])

**Insights**- **Reading — the fan-out inflates revenue by 58.5%.** Summing the raw `amount` column gives  **\$111,285**; collapsing the 3,260 payment rows to their **2,425** distinct charges gives  **\$70,229**. Every per-member and per-site dollar figure in this notebook uses the latter.- **Reading — the wash side is nearly clean:** 7,741 rows → **7,492** vehicle-washes, i.e. only 249  duplicate rows (3.2%), all of them one vehicle logged under two plate spellings.- **So-what — household size is a feature, not noise.** The fan-out width *is* the number of  vehicles on the account at the time of the charge, so collapsing it preserves the signal as  `n_vehicles_billed`. It turns out to be the strongest protective factor in §5.- **Caveat:** 96 of 466 customers have 2+ vehicles, so the correction is concentrated — a naive  analysis would not be uniformly 58% wrong, it would be wrong *specifically about the best  customers*, which is worse.

---## 2. Data-quality auditWhat is actually populated, and what has to be thrown away before modelling.

In [ ]:
q = pd.DataFrame({
    "null %": (raw.isna().mean() * 100).round(1),
    "distinct": raw.nunique(),
}).sort_values("null %", ascending=False)
q["verdict"] = np.where(q["null %"] > 90, "DROP - unusable",
                 np.where(q["null %"] > 15, "partial - use with care", "usable"))
q

**Insights**- **Reading — three vehicle columns are unusable and are dropped:** `vehicle_vin` (100% null),  `vehicle_year` (99.6%), `vehicle_model` (96.1%). Any "profile by vehicle age / model" ambition  dies here.- **Reading — the 70.4% nulls on `amount` / `payment_type` / `current_package_price` are  structural, not missing data:** those columns are only populated on `payment` rows, and 70.4% of  rows are washes. Correctly read, they are 100% complete.- **So-what — the usable profiling attributes are behavioural, not demographic.** `vehicle_make` is  17.7% null and `vehicle_state` is 82.7% a single value (KY), so the only real discriminators in  this file are *what members pay* and *how often they wash*. That is what §7 segments on.- **Caveat:** `vehicle_type` is 2.9% null and dominated by "Passenger Vehicle" (319 of 411  classified members) — the Bus/Truck/Work Van cells are n≤4 and are anecdotes, not segments.

In [ ]:
cust = P.customer_table(ev)
issues = pd.DataFrame([
    ("exact duplicate rows dropped", len(raw) - len(ev), "removed at load"),
    ("customers with washes but no payment row", int(cust.cycles_paid.isna().sum()),
     "customer 101 - excluded from economics"),
    ("charges with no vehicle_id on file", int(pay.no_vehicle_on_file.sum()),
     "flagged, not imputed"),
    ("$0 renewals (comped months)", int(pay.is_comp.sum()), "kept - member is still live"),
    ("negative amounts (refunds)", int(pay.is_refund.sum()),
     f"${pay[pay.is_refund].amount.sum():,.0f}, {pay[pay.is_refund].customer_id.nunique()} members"),
], columns=["issue", "count", "handling"])
issues

**Insights**- **Reading — the file is in good shape:** the five defects together touch ~150 of 11,001 rows.  Only one (customer 101, washes but no payments) forces an exclusion.- **So-what — `$0` renewals are members, not errors.** All 15 are comped months on live accounts;  dropping them would fabricate 15 churn events. They stay in the panel as cycles with  `amount = 0`.- **So-what — refunds are netted, not deleted.** 64 negative charges across 44 members total  **−\$1,048** (1.5% of revenue). They are excluded from the *cycle* count (a refund is not a  membership month) but included in revenue.- **Caveat:** 38 charges carry no `vehicle_id` at all and renew at just **63.2%** vs 93.0%. That  is a real signal but a confounded one — "never registered a car" plausibly means "never intended  to use it", so it gets its own flag (`no_vehicle_on_file`) rather than being folded into  household size.

---## 3. The book at a glanceThis is a **young site ramping up**, not a steady-state book — the first payment is 2025-09-13 andthe export ends 330 days later. Every retention number below has to be read with that in mind.

In [ ]:
joins = cust.groupby("cohort").size().rename("joined")
lost = cust[cust.churned].groupby("churn_month").size().rename("churned")
flow = pd.concat([joins, lost], axis=1).fillna(0).astype(int).sort_index()
flow["net"] = flow.joined - flow.churned

fig = go.Figure()
fig.add_bar(x=flow.index, y=flow.joined, name="Joined", marker_color=T.s1,
            hovertemplate="%{x}<br>%{y} joined<extra></extra>")
fig.add_bar(x=flow.index, y=-flow.churned, name="Churned", marker_color=T.s2,
            hovertemplate="%{x}<br>%{customdata} churned<extra></extra>", customdata=flow.churned)
fig.add_scatter(x=flow.index, y=flow.net, name="Net change", mode="lines+markers",
                line=dict(color=T.ink, width=2), marker=dict(size=8),
                hovertemplate="%{x}<br>net %{y:+d}<extra></extra>")
viz.style(fig, T, barmode="relative", height=380,
          title=dict(text="Members joined and lost each month"),
          yaxis=dict(title="members"), xaxis=dict(title=""))
fig.add_hline(y=0, line_color=T.axis, line_width=1)
fig.show()
flow.T

**Insights**- **Reading — acquisition has stalled while churn has not.** Signups peaked at **105 in Dec 2025**  and fell to **13 in Jun** and **23 in Jul 2026**, while monthly churn settled at **17–19  members**. The book went net-negative in **Jun 2026 (−6)** for the first time.- **So-what — the growth problem is now retention, not acquisition.** At 334 active members and  ~18 lost per month, the site must sign ~18/month just to stand still; it signed 13 in June.- **Caveat — the Aug 2026 column is a partial month** (the export stops on the 10th) and its zero  churns are an artefact of the 40-day observation rule, not a turnaround. Jan 2026's 18 churns are  the December cohort's promo expiring — see §5b.

In [ ]:
mw = wsh.set_index("event_date").resample("MS").size().rename("washes")
mc = (pay[~pay.is_refund].set_index("event_date").resample("MS").size().rename("paid cycles"))
act = pd.concat([mw, mc], axis=1).fillna(0).astype(int)

fig = go.Figure()
fig.add_scatter(x=act.index, y=act["washes"], name="Washes", mode="lines",
                line=dict(color=T.s1, width=2), hovertemplate="%{x|%b %Y}<br>%{y} washes<extra></extra>")
fig.add_scatter(x=act.index, y=act["paid cycles"], name="Paid cycles", mode="lines",
                line=dict(color=T.s3, width=2), hovertemplate="%{x|%b %Y}<br>%{y} cycles<extra></extra>")
viz.style(fig, T, height=360, title=dict(text="Monthly volume: washes vs paid membership months"),
          yaxis=dict(title="count"), xaxis=dict(title=""))
fig.show()

summary = pd.Series({
    "members (ever)": len(cust),
    "active at 2026-08-10": int(cust.active.sum()),
    "churned": int(cust.churned.sum()),
    "lifetime churn rate": f"{cust.churned.mean():.1%}",
    "net revenue": f"${pay.amount.sum():,.0f}",
    "active-book MRR": f"${cust[cust.active].arpu.sum():,.0f}",
    "washes delivered": len(wsh),
    "washes per paid cycle": round(len(wsh) / (~pay.is_refund).sum(), 2),
}, name="value").to_frame()
summary

**Insights**- **Reading — utilisation intensity is flat, so the wash curve is pure member growth.** Both series  plateau together from Apr 2026 (~1,100 washes, ~450 cycles); the ratio holds near **3.1 washes  per paid membership month** across the window.- **So-what — at \$9,726 MRR from 334 active members, ARPU is ~\$29** against list prices of  \$18.99–\$39. The gap is the promo book (§5d), not a discounting policy.- **Caveat:** Aug 2026 is 10 days of a month and drops on both series — read the last point as  incomplete, not as a collapse.

---## 4. Retention: cohorts and the renewal hazardTwo views of the same thing. The **cohort table** answers "how many of the people who joined inmonth X are still here?"; the **hazard curve** answers "at which membership month do we losethem?". The second is the actionable one.Both respect censoring: a member who joined three weeks ago has not *survived* three weeks, theyare simply unobserved, and never counts in a denominator.

In [ ]:
ret = P.cohort_retention(ev)
z = ret.values.astype(float)
fig = go.Figure(go.Heatmap(
    z=z, x=[f"M{c}" for c in ret.columns], y=ret.index,
    colorscale=[[i / (len(T.seq) - 1), c] for i, c in enumerate(T.seq)],
    zmin=0, zmax=1, xgap=2, ygap=2, hoverongaps=False,
    colorbar=dict(title="retained", tickformat=".0%", outlinewidth=0,
                  tickfont=dict(color=T.muted)),
    hovertemplate="%{y} cohort, %{x}<br>%{z:.0%} retained<extra></extra>"))
for i, coh in enumerate(ret.index):
    for j, c in enumerate(ret.columns):
        v = ret.iloc[i, j]
        if pd.notna(v):
            fig.add_annotation(x=j, y=i, text=f"{v:.0%}", showarrow=False,
                               font=dict(size=9, color="#ffffff" if v > 0.55 else T.ink))
# type="category" matters: the cohort labels are strings like "2025-09", which Plotly would
# otherwise parse as dates and render on a 1970-based time axis.
viz.style(fig, T, height=420, title=dict(text="Cohort retention by months since signup"),
          xaxis=dict(title="months since signup", showgrid=False, type="category"),
          yaxis=dict(title="signup cohort", showgrid=False, type="category", autorange="reversed"))
fig.show()

**Insights**- **Reading — the first month is the cliff.** Every cohort drops hardest between M0 and M1  (2025-09: 100%→73%, 2025-12: 100%→86%, 2026-05: 100%→86%). After M1 the slope flattens to  roughly 3–6 points per month.- **Reading — later cohorts start better but converge.** The Jan 2026 cohort held **97% at M1** and  still fell to **57% by M7**; the Sep 2025 cohort reached **36% at M10**. Nothing yet suggests a  cohort that escapes the decay, only ones that start higher.- **So-what — retention spend belongs in the first 30–60 days.** Two thirds of the eventual loss in  the oldest cohort had already happened by M5.- **Caveat — the bottom-right of the table is empty by construction**, and the bottom-left cells  are thin: the 2026-07 cohort's **60% at M1** is 23 members, and 2026-06's **100% at M2** is 13.  Cells under ~20 members should not be read as trend.

In [ ]:
panel = P.renewal_panel(ev)
haz = P.hazard_curve(panel)
o = panel[~panel.censored]

fig = go.Figure()
fig.add_scatter(x=haz.index, y=haz.renewal_rate, mode="lines+markers",
                line=dict(color=T.s1, width=2), marker=dict(size=9),
                customdata=haz.n, hovertemplate="month %{x}<br>%{y:.1%} renew (n=%{customdata})<extra></extra>",
                showlegend=False)
fig.add_hline(y=o.renewed.mean(), line_dash="dot", line_color=T.muted,
              annotation_text=f"book average {o.renewed.mean():.1%}",
              annotation_font=dict(color=T.muted, size=11))
for x, r, n in zip(haz.index, haz.renewal_rate, haz.n):
    fig.add_annotation(x=x, y=r, text=f"n={n}", yshift=-18, showarrow=False,
                       font=dict(size=9, color=T.muted))
viz.style(fig, T, height=380, title=dict(text="Renewal rate by membership month (censored cycles excluded)"),
          yaxis=dict(title="renewed next cycle", tickformat=".0%", range=[0.82, 1.02]),
          xaxis=dict(title="membership month", dtick=1))
fig.show()
haz.round(3)

**Insights**- **Reading — month 0 is where the book leaks.** The signup month renews at **87.6% (n=437)**  against **93.8% (n=1,590)** for every later month — a 6.2-point gap, χ² **p = 2.7e-05**. In  absolute terms that one month alone accounts for 54 of the 153 observed cycle churns.- **Reading — survivors get stickier, monotonically to M6:** 92.3% → 92.9% → 93.5% → 96.9% →  **99.2% at M6**. This is selection, not loyalty growth: the people who dislike it have left.- **So-what — the book's steady-state churn is much better than its headline.** Excluding month 0,  monthly churn is **6.2%**, implying a ~16-month lifetime for anyone who clears the first renewal.- **Caveat — M7+ is thin and non-monotonic** (M7 = 91.7% on n=60, M9 = 100% on n=16). The dip at  M7 is 5 churns; do not read it as a second cliff. 334 of 2,361 cycles are censored and excluded  throughout.

---## 5. What drives churnFive hypotheses, each tested on the renewal panel — one row per paid membership month, withfeatures measured **at the moment of the charge** and the label being whether the *next* chargearrived. Censored cycles are excluded from every rate.

In [ ]:
def rate_by(frame, bucket, label):
    g = frame.groupby(bucket, observed=True).agg(n=("renewed", "size"), renewal=("renewed", "mean"))
    return g.rename_axis(label)


b = pd.cut(o.washes_this_cycle, [-1, 0, 1, 2, 4, 8, 1000], labels=["0", "1", "2", "3-4", "5-8", "9+"])
dorm = rate_by(o, b, "washes in the cycle just paid for")

fig = go.Figure(go.Bar(
    x=dorm.index.astype(str), y=dorm.renewal, marker_color=[T.s2] + [T.s1] * 5,
    text=[f"{v:.1%}" for v in dorm.renewal], textposition="outside",
    textfont=dict(color=T.ink2), customdata=dorm.n,
    marker_line=dict(width=2, color=T.surface),
    hovertemplate="%{x} washes<br>%{y:.1%} renew (n=%{customdata})<extra></extra>"))
viz.style(fig, T, height=380, showlegend=False,
          title=dict(text="a) Dormancy — members who did not wash are the ones who leave"),
          yaxis=dict(title="renewal rate", tickformat=".0%", range=[0.8, 1.0]),
          xaxis=dict(title="washes in the 30 days ending at the charge"))
fig.show()

chi = stats.chi2_contingency(pd.crosstab(o.dormant, o.renewed))
print(f"zero-wash {o[o.dormant].renewed.mean():.1%} (n={o.dormant.sum()})  vs  "
      f"any wash {o[~o.dormant].renewed.mean():.1%} (n={(~o.dormant).sum()})   chi2 p={chi[1]:.2g}")
print(f"dormant share of all paid cycles: {o.dormant.mean():.1%}  "
      f"= ${o[o.dormant].amount.sum():,.0f} collected ({o[o.dormant].amount.sum()/o.amount.sum():.1%} of cycle revenue)")
dorm.round(3)

**Insights**- **Reading — dormancy is the single biggest lever and the dose-response is monotone:** 0 washes  **87.0%** → 1 **91.7%** → 2 **93.8%** → 3-4 **95.4%** → 5-8 **96.0%**, flattening at 9+ (95.8%).  Zero-wash vs any-wash is **87.0% vs 94.7%**, χ² **p = 4e-09** on n=2,027.- **So-what — 29.2% of all paid membership months involved no wash at all**, worth **\$8,685**  (14.5% of cycle revenue). That is simultaneously the most profitable revenue on the book (no  wash cost) and the most fragile.- **So-what — the intervention is obvious and cheap:** a "you haven't been in" nudge fired at day  ~20 of a dormant cycle targets a group whose renewal rate is 7.7 points below everyone else's.- **Caveat — the direction of causation is not settled here.** Not washing may cause cancelling, or  a member who has already decided to quit simply stops coming. The gap is real either way, but the  *uplift* from a nudge is untested — it needs a holdout, which this export cannot supply.

In [ ]:
oo = o[o.prev_amount.notna()]
step = pd.DataFrame({
    "renewal": [oo[oo.price_step_up].renewed.mean(), oo[~oo.price_step_up].renewed.mean()],
    "n": [int(oo.price_step_up.sum()), int((~oo.price_step_up).sum())],
}, index=["price stepped up >15%", "price held flat"])
c2 = stats.chi2_contingency(pd.crosstab(oo.price_step_up, oo.renewed))

fig = go.Figure(go.Bar(
    x=step.index, y=step.renewal, marker_color=[T.s2, T.s1],
    text=[f"{v:.1%}" for v in step.renewal], textposition="outside", textfont=dict(color=T.ink2),
    customdata=step.n, marker_line=dict(width=2, color=T.surface),
    hovertemplate="%{x}<br>%{y:.1%} renew (n=%{customdata})<extra></extra>"))
viz.style(fig, T, height=340, showlegend=False,
          title=dict(text="b) The promo cliff — what happens when the intro price ends"),
          yaxis=dict(title="renewal rate", tickformat=".0%", range=[0.85, 0.98]),
          xaxis=dict(title=""))
fig.show()

first_step = oo[oo.price_step_up].groupby("customer_id").cycle_no.min()
print(f"step-up {step.renewal.iloc[0]:.1%} vs flat {step.renewal.iloc[1]:.1%}   chi2 p={c2[1]:.3g}")
print(f"the step lands at membership month 1 for {first_step.eq(1).sum()} of {len(first_step)} members")
print(f"typical jump: ${oo[oo.price_step_up].prev_amount.median():.0f} -> "
      f"${oo[oo.price_step_up].amount.median():.0f}  ({oo[oo.price_step_up].price_ratio.median():.1f}x)")

**Insights**- **Reading — the intro promo ends at month 1 and costs 4.8 points of renewal.** Charges that jump  >15% renew at **90.2% (n=410)** vs **95.0% (n=1,180)** for flat charges, χ² **p = 0.0009**. For  **333 of 351** affected members the jump lands at membership month 1.- **Reading — the jump is not marginal, it is 3.2×:** the median member goes **\$10 → \$32** in one  step.- **So-what — this explains the month-0 cliff in §4.** The signup month's 87.6% renewal and the  price step are the same event seen twice; the cliff is a *pricing* artefact, not a service one.  A two-step ramp (\$10 → \$20 → \$32) is the obvious thing to A/B test.- **Caveat — step-up and month-1 are almost perfectly collinear** (95% of steps happen at M1), so  this design cannot separate "price shock" from "the natural month-1 drop-off". The model in §6  includes both and still gives the step an independent 1.27× odds effect, but that separation  rests on the 5% of steps that land later — treat it as suggestive.

In [ ]:
onv = o[~o.no_vehicle_on_file]
veh = rate_by(onv, onv.n_vehicles_billed.clip(upper=4), "vehicles on the account")
veh.index = ["1", "2", "3", "4+"]
c3 = stats.chi2_contingency(pd.crosstab(onv.n_vehicles_billed > 1, onv.renewed))

fig = go.Figure(go.Bar(
    x=veh.index, y=veh.renewal, marker_color=[T.s2] + [T.s1] * 3,
    text=[f"{v:.1%}" for v in veh.renewal], textposition="outside", textfont=dict(color=T.ink2),
    customdata=veh.n, marker_line=dict(width=2, color=T.surface),
    hovertemplate="%{x} vehicles<br>%{y:.1%} renew (n=%{customdata})<extra></extra>"))
viz.style(fig, T, height=340, showlegend=False,
          title=dict(text="c) Household size — the strongest protective factor in the book"),
          yaxis=dict(title="renewal rate", tickformat=".0%", range=[0.88, 1.02]),
          xaxis=dict(title="vehicles billed on the account"))
fig.show()
print(f"1 vehicle {onv[onv.n_vehicles_billed==1].renewed.mean():.1%} vs 2+ "
      f"{onv[onv.n_vehicles_billed>1].renewed.mean():.1%}   chi2 p={c3[1]:.2g}")
veh.round(3)

**Insights**- **Reading — adding a second car to the account halves the churn rate.** One vehicle renews at  **92.0% (n=1,542)**, two or more at **96.4% (n=447)**, χ² **p = 0.0019**. The gradient continues  (3 cars **96.5%**, 4+ **100%** on n=42).- **So-what — this is the one lever the operator fully controls and can act on today.** Converting  a single-car member to a two-car plan is worth ~5 points of monthly retention *and* raises ARPU  from ~\$27 to ~\$45 — it is the highest-value cross-sell in the file.- **Caveat — selection is doing some of this work.** Households that add a second car were probably  more committed to begin with; the file cannot tell you whether *offering* the second car creates  the loyalty or merely reveals it. The 4+ cell is n=42 and its 100% should be read as "very high",  not as literal.

In [ ]:
c4 = stats.chi2_contingency(pd.crosstab(o.joined_on_promo, o.renewed))
promo = pd.DataFrame({
    "renewal": [o[o.joined_on_promo].renewed.mean(), o[~o.joined_on_promo].renewed.mean()],
    "n": [int(o.joined_on_promo.sum()), int((~o.joined_on_promo).sum())],
}, index=["joined on promo", "joined at full price"])
print(promo.round(3).to_string())
print(f"\nchi2 p = {c4[1]:.2g}  -- NOT significant")
print(f"promo share of all signups: {cust.joined_on_promo.mean():.1%}")

**Insights**- **Reading — a null result, and a useful one.** Promo joiners renew at **92.3% (n=1,859)**,  full-price joiners at **94.0% (n=168)**; χ² **p = 0.51**. The 1.7-point gap is noise.- **So-what — "discount hunters are bad customers" is not supported here.** *Who* signed up on a  promo tells you nothing; what predicts churn is what they do afterwards (§5a) and what happens to  their price (§5b). The promo is not attracting a worse class of member — it is creating a price  shock 30 days later.- **Caveat — this test is badly underpowered and should not be over-read.** 91.4% of signups took  the promo, leaving only **168 full-price cycles** as a comparison group; the design can only rule  out a large effect, not a modest one. It is also observational — full-price joiners are  self-selected.

In [ ]:
pkg = (o.groupby("membership_package_name")
        .agg(n=("renewed", "size"), renewal=("renewed", "mean"),
             list_price=("current_package_price", "first"),
             washes=("washes_this_cycle", "mean"))
        .sort_values("n", ascending=False))
pkg.round(3)

**Insights**- **Reading — package choice barely matters, and it is not ordered by price.** Among the four with  real volume: Typhoon (\$32) **95.0%**, Big Kahuna (\$25) **92.4%**, Monsoon (\$39) **91.8%**,  Little Kahuna (\$18.99) **90.5%**. The *cheapest* plan retains worst and the mid plan best, so  there is no price-sensitivity gradient here.- **So-what — don't build the retention story on tier.** The 4.5-point spread between Typhoon and  Little Kahuna is smaller than the dormancy effect (7.7 points) and smaller than the household-size  effect (5.1 points), both of which are cleanly significant.- **Caveat — the 1st-Responder variants are n=7 to 43** and their 95–100% renewal is an anecdote.  Tier is also confounded with wash entitlement, which the export does not describe.

---## 6. A churn model**Frame:** discrete-time renewal hazard. One row per paid membership month; the label is whetherthe next charge arrived within 45 days; features are only what was knowable at the moment of thecharge. This is the frame that handles a ramping book correctly — a customer-level "will they everchurn" model on a site this young would mostly learn *when someone joined*.**Why logistic regression:** 2,027 usable rows and a 7.5% event rate is not enough to feed a treeensemble, and the coefficients are half the deliverable. A LightGBM check below confirms it is notleaving signal on the table.

In [ ]:
mod = P.fit_churn_model(panel)
print(f"train {mod.n_train} cycles  ->  holdout {mod.n_test} cycles after {mod.cutoff.date()}")
print(f"AUC   5-fold CV {mod.auc_cv:.3f}   |   time-ordered holdout {mod.auc_holdout:.3f}")
print(f"base churn per cycle {mod.base_rate:.1%}   |   top-decile lift {mod.top_decile_lift:.2f}x")

# Is a booster finding anything the linear model misses?
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

d = P._model_frame(panel[~panel.censored])
X, y = d[P.FEATURES], d.renewed.astype(int)
aucs = []
for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(X, y):
    g = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=7,
                           min_child_samples=40, verbose=-1)
    g.fit(X.iloc[tr], y.iloc[tr])
    aucs.append(roc_auc_score(y.iloc[te], g.predict_proba(X.iloc[te])[:, 1]))
print(f"LightGBM 5-fold CV AUC {np.mean(aucs):.3f}  (vs logistic {mod.auc_cv:.3f})")

**Insights**- **Reading — the model ranks meaningfully: holdout AUC 0.713**, on a *time-ordered* split (train  on cycles up to 2026-05-23, predict the 507 after) which is strictly harder than random CV and  matches how it would be deployed.- **Reading — LightGBM finds nothing extra** (CV AUC 0.662 vs logistic 0.679), so there are no  important interactions or non-linearities being missed. The linear model is the right complexity  for this data.- **So-what — 2.71× top-decile lift is the operational number:** the riskiest 10% of member-months  churn at 2.7× the base rate, which is what makes a targeted save campaign cheaper than a blanket  one.- **Caveat — AUC ~0.71 is useful for triage, not for individual verdicts.** Most of the signal is  recency and price-step; the file has no service-quality, complaint, or competitor data, which is  plausibly where the rest lives.

In [ ]:
orat = mod.odds_ratios()
colors = [T.critical if v > 1 else T.good for v in orat]
fig = go.Figure(go.Bar(
    x=orat.values - 1, y=orat.index, orientation="h", marker_color=colors,
    marker_line=dict(width=2, color=T.surface),
    text=[f"{v:.2f}x" for v in orat], textposition="outside", textfont=dict(color=T.ink2),
    hovertemplate="%{y}<br>%{text} churn odds per +1 SD<extra></extra>"))
viz.style(fig, T, height=420, showlegend=False,
          title=dict(text="Churn odds multiplier per +1 SD  (right = raises churn)"),
          xaxis=dict(title="odds multiplier", tickvals=[-0.4, -0.2, 0, 0.2, 0.4],
                     ticktext=["0.6x", "0.8x", "1.0x", "1.2x", "1.4x"]),
          yaxis=dict(title="", autorange="reversed"))
fig.add_vline(x=0, line_color=T.axis, line_width=1)
fig.show()
orat.round(2).to_frame("churn odds per +1 SD")

**Insights**- **Reading — recency dominates:** `days_since_wash` at **1.37×** churn odds per SD, ahead of  `price_step_up` **1.27×** and `no_vehicle_on_file` **1.28×**. Everything the univariate tests in  §5 found survives being controlled for the others.- **Reading — `n_vehicles_billed` at 0.69× is the strongest protective term**, confirming §5c  independently of tenure and spend.- **So-what — the two actionable levers are re-engagement and the price ramp**, in that order.  Both are operational changes, not pricing-strategy changes.- **Caveat — `joined_on_promo` at 0.98× is indistinguishable from no effect**, consistent with the  §5d null. `month` at 1.19× is *not* a seasonality finding: with only 11 months of a ramping book  it is absorbing calendar time, and should not be extrapolated.

In [ ]:
risk = mod.score(d)
q = pd.qcut(risk, 5, labels=["Q1 safest", "Q2", "Q3", "Q4", "Q5 riskiest"])
cal = (pd.DataFrame({"pred": risk, "actual": 1 - y.values})
       .groupby(q, observed=True).agg(n=("actual", "size"), predicted=("pred", "mean"),
                                      actual=("actual", "mean")))

fig = go.Figure()
fig.add_bar(x=cal.index.astype(str), y=cal.predicted, name="Predicted churn", marker_color=T.s1,
            marker_line=dict(width=2, color=T.surface),
            hovertemplate="%{x}<br>predicted %{y:.1%}<extra></extra>")
fig.add_bar(x=cal.index.astype(str), y=cal.actual, name="Actual churn", marker_color=T.s2,
            marker_line=dict(width=2, color=T.surface),
            hovertemplate="%{x}<br>actual %{y:.1%}<extra></extra>")
viz.style(fig, T, height=360, barmode="group",
          title=dict(text="Calibration — predicted vs realised churn by risk quintile"),
          yaxis=dict(title="churn rate", tickformat=".0%"), xaxis=dict(title=""))
fig.show()
cal.round(3)

**Insights**- **Reading — the model is calibrated, not just discriminative.** Predicted vs actual tracks across  the range: Q1 **2.6% / 2.5%**, Q2 **4.7% / 4.4%**, Q5 **15.8% / 15.3%**. That is what allows the  risk score to be multiplied by ARPU and reported in dollars.- **Reading — the riskiest quintile churns 6× the safest** (15.3% vs 2.5%).- **So-what — Q5 is 406 member-months carrying 62 of the 153 observed churns**; a save offer aimed there  reaches 40% of the loss while touching 20% of the book.- **Caveat — Q3 is the one soft spot** (predicts 5.9%, delivers 4.7% on n=405) and these are  in-sample fitted values; the honest out-of-sample evidence is the 0.713 holdout AUC above, not  this chart.

---## 7. Segmentation — four personasK-means on six standardised behaviour-and-economics features (`washes_per_month`, `tenure_months`,`arpu`, `n_vehicles`, `cost_per_wash`, `days_since_wash`). Money features are log-transformed;personas are named by matching each centroid to an archetype via optimal assignment, so a re-fitcannot silently swap two labels.

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

sil = {}
for k in range(2, 7):
    dd, _ = P.segment(cust, k=k)
    Xs = dd[P.SEG_FEATURES].copy()
    Xs["days_since_wash"] = Xs.days_since_wash.fillna(120).clip(upper=120)
    for c in ["arpu", "cost_per_wash", "washes_per_month"]:
        Xs[c] = np.log1p(Xs[c].clip(lower=0))
    sil[k] = silhouette_score(StandardScaler().fit_transform(Xs), dd.segment_id)
print("silhouette by k:", {k: round(v, 3) for k, v in sil.items()})

seg, prof = P.segment(cust, k=4)
prof[["members", "washes_per_month", "tenure_months", "arpu", "n_vehicles",
      "cost_per_wash", "days_since_wash", "churn_rate", "revenue_share"]].round(2)

**Insights**- **Reading — four personas, and they are economically very unlike each other.** Churn rate runs  from **5%** (Core regular) to **74%** (Promo flipper); ARPU from **\$10** to **\$47**.- **Reading — 73 Power households (16% of members) are 35% of revenue.** They wash **5.05×/month**  across **2 vehicles**, pay **\$47**, and churn at 10%.- **So-what — "Never activated" is the most addressable group:** 93 members paying **\$19.29** for  **0.68 washes/month** (\$20 per wash actually taken) who then churn at **58%**. They are paying  and not receiving value — a pure onboarding failure, and 11% of revenue.- **Caveat — k=4 (silhouette 0.304) is a judgement call, not a discovered truth.** k=5 scores  marginally higher (0.309) and k=2 nearly as well (0.299); the clusters are real but the *number*  of them is soft. Personas are also computed on lifetime aggregates, so tenure partly encodes  when someone joined.

In [ ]:
order = [s for s in viz.SEGMENT_ORDER if s in prof.index]
fig = go.Figure()
fig.add_bar(x=order, y=[prof.loc[s, "members"] / prof.members.sum() for s in order],
            name="Share of members", marker_color=T.s1, marker_line=dict(width=2, color=T.surface),
            hovertemplate="%{x}<br>%{y:.0%} of members<extra></extra>")
fig.add_bar(x=order, y=[prof.loc[s, "revenue_share"] for s in order],
            name="Share of revenue", marker_color=T.s2, marker_line=dict(width=2, color=T.surface),
            hovertemplate="%{x}<br>%{y:.0%} of revenue<extra></extra>")
viz.style(fig, T, height=380, barmode="group",
          title=dict(text="Who they are vs what they are worth"),
          yaxis=dict(title="share", tickformat=".0%"), xaxis=dict(title=""))
fig.show()

**Insights**- **Reading — revenue is far more concentrated than headcount.** Power households are **16% of  members / 35% of revenue**; Promo flippers are **17% of members / 2% of revenue** — a ~10×  over-representation at one end and ~8× under at the other.- **So-what — a per-member retention budget is the wrong unit.** Saving one Power household is  worth roughly eight Promo flippers, and the model in §6 already scores them (Power households  average 3.2% risk vs 12.8% for flippers).- **Caveat — revenue share is cumulative over the window**, so it rewards tenure mechanically:  Power households have been on the book ~7 months, flippers ~1. On an MRR basis the gap narrows,  though it does not close.

In [ ]:
# Four personas in a scatter would need an all-pairs-safe 4-colour palette, which the validated
# palette does not provide -- so this is small multiples, one persona per panel, shared axes.
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=4, subplot_titles=order, shared_yaxes=True, horizontal_spacing=0.02)
for i, s in enumerate(order, start=1):
    sub = seg[seg.segment == s]
    fig.add_scatter(x=sub.washes_per_month, y=sub.arpu, mode="markers", name=s,
                    marker=dict(color=SEG_COLOR[s], size=8, opacity=0.75,
                                line=dict(width=2, color=T.surface)),
                    customdata=np.stack([sub.tenure_months, sub.n_vehicles], axis=-1),
                    hovertemplate=("%{fullData.name}<br>%{x:.1f} washes/mo<br>$%{y:.0f} ARPU"
                                   "<br>%{customdata[0]:.1f} mo tenure, %{customdata[1]:.0f} car(s)"
                                   "<extra></extra>"),
                    row=1, col=i, showlegend=False)
viz.style(fig, T, height=340,
          title=dict(text="Usage vs spend, one panel per persona (shared axes)"))
fig.update_xaxes(gridcolor=T.grid, linecolor=T.axis, tickfont=dict(color=T.muted),
                 range=[-0.5, 14], title_text="washes / month", title_font=dict(size=11))
fig.update_yaxes(gridcolor=T.grid, linecolor=T.axis, tickfont=dict(color=T.muted), range=[0, 130])
fig.update_yaxes(title_text="ARPU ($)", row=1, col=1)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=T.ink)
fig.show()

**Insights**- **Reading — the personas separate on two different axes, not one.** Power households spread  along *both* usage and spend (up to ~13 washes/month and \$100+ ARPU); Promo flippers collapse  into a single point at **\$10 ARPU** regardless of usage — several wash 5+ times a month while  paying \$10.- **Reading — Never-activated is a vertical stripe near x=0:** normal spend, no usage. Visually  the clearest failure mode in the book.- **So-what — Promo flippers are extracting real value at \$3.33 per wash** and still leaving,  which says the \$10 price is not the reason they stay, and the \$32 price is the reason they go.  This is the same promo-cliff finding from a different direction.- **Caveat — ARPU is an average over a member's cycles**, so a flipper who paid \$10 once sits at  exactly \$10 by construction; the tight cluster is partly definitional.

---## 8. Unit economics and CLVContribution = ARPU − (washes/month × variable cost per wash). CLV = monthly contribution ÷monthly churn, the standard geometric-series lifetime.**The variable cost per wash is an assumption, not data** — this export has no cost side. \$2.25(water, chemicals, power, incremental labour) is a plausible express-tunnel figure and thesensitivity below shows how much rides on it.

In [ ]:
churn_m = P.observed_monthly_churn(cust)
ue = P.unit_economics(seg, variable_cost_per_wash=2.25)
econ = ue.groupby("segment").agg(
    members=("customer_id", "size"), arpu=("arpu", "median"),
    wash_cost=("monthly_wash_cost", "median"),
    contribution=("monthly_contribution", "median"), clv=("clv", "median")).reindex(order)
print(f"observed monthly churn {churn_m:.2%}  ->  implied average lifetime {1/churn_m:.1f} months")

fig = go.Figure(go.Bar(
    x=econ.index, y=econ.clv, marker_color=[SEG_COLOR[s] for s in econ.index],
    marker_line=dict(width=2, color=T.surface),
    text=[f"${v:,.0f}" for v in econ.clv], textposition="outside", textfont=dict(color=T.ink2),
    customdata=np.stack([econ.arpu, econ.contribution], axis=-1),
    hovertemplate="%{x}<br>CLV $%{y:,.0f}<br>ARPU $%{customdata[0]:.0f}, "
                  "contribution $%{customdata[1]:.0f}/mo<extra></extra>"))
viz.style(fig, T, height=360, showlegend=False,
          title=dict(text="Median CLV per persona (at $2.25 variable cost per wash)"),
          yaxis=dict(title="CLV ($)"), xaxis=dict(title=""))
fig.show()
econ.round(2)

**Insights**- **Reading — a Power household is worth 10× a Promo flipper:** median CLV **\$586** vs **\$56**,  driven by both a higher contribution (**\$33.73** vs **\$3.25**/month) and lower churn.- **Reading — "Never activated" members are quietly profitable:** \$19.29 ARPU against \$1.53 of  wash cost = **\$17.49/month contribution**, nearly matching Core regulars (\$18.95) at a third of  the delivery cost. Their problem is the 58% churn rate, not the margin.- **So-what — that creates a genuine tension worth naming.** Waking up a dormant member converts  high-margin revenue into low-margin revenue; it is only worth doing if the retention gain  (7.7 points, §5a) outweighs the added wash cost. At \$2.25/wash and \$29 ARPU it does, but the  margin is not enormous.- **Caveat — CLV uses one book-wide churn rate (5.8%/month)** rather than a per-persona rate, so  the spread across personas is driven by contribution alone and is *understated*. Using each  persona's own churn would widen the Power-vs-flipper gap considerably.

In [ ]:
sens = pd.DataFrame([
    {"$/wash": vc,
     "unprofitable members": int((P.unit_economics(seg, vc).monthly_contribution < 0).sum()),
     "median CLV": P.unit_economics(seg, vc).clv.median(),
     "book contribution/mo": P.unit_economics(seg, vc).query("active").monthly_contribution.sum()}
    for vc in [1.00, 1.50, 2.25, 3.00, 4.00, 5.00]])

fig = go.Figure()
fig.add_scatter(x=sens["$/wash"], y=sens["book contribution/mo"], mode="lines+markers",
                line=dict(color=T.s1, width=2), marker=dict(size=9), showlegend=False,
                hovertemplate="$%{x:.2f}/wash<br>$%{y:,.0f}/mo contribution<extra></extra>")
viz.style(fig, T, height=340,
          title=dict(text="Active-book monthly contribution vs the cost assumption"),
          yaxis=dict(title="contribution ($/month)"), xaxis=dict(title="variable cost per wash ($)"))
fig.show()
sens.round(2)

**Insights**- **Reading — the conclusion is robust across the plausible cost range.** From \$1.00 to \$5.00 per  wash the active book stays firmly contribution-positive; median CLV moves \$372 → \$214, a wide  band but never a sign change.- **Reading — the count of unprofitable members is the sensitive number:** 10 at \$1.00/wash rising  to **80 at \$5.00**. Heavy users are the ones who flip.- **So-what — before running a "wash more!" campaign, pin down the real marginal cost.** The  retention case for re-engagement (§5a) holds at any cost in this range, but the *profitability*  of driving extra volume from already-heavy users does not.- **Caveat — this is a one-parameter sensitivity on an assumption with no data behind it**, and it  ignores fixed cost entirely, so "contribution" here is not profit.

---## 9. What this dataset can and cannot answer### Problem statements it supports today| # | Problem statement | Evidence | Status ||---|---|---|---|| 1 | **Which members will cancel next month?** | Calibrated hazard model, holdout AUC 0.713, 2.71× top-decile lift | Ready — drives the demo's risk list || 2 | **Is the intro promo structured wrong?** | Step-up costs 4.8 pts of renewal (p=0.0009); 3.2× jump at month 1 for 333/351 members | Ready — proposes a 2-step ramp to A/B test || 3 | **Who should retention spend target?** | Dormancy 7.7 pts (p=4e-09); Q5 risk quintile = 40% of churn in 20% of the book | Ready || 4 | **What is a member worth, by type?** | 4 personas, CLV \$56–\$586; 16% of members = 35% of revenue | Ready, modulo the cost assumption || 5 | **Is multi-car the best cross-sell?** | 2+ vehicles renew 5.1 pts higher (p=0.0019) and pay ~\$18 more | Ready as a hypothesis; causality untested || 6 | **Are we billing people who never come?** | 29.2% of paid cycles had zero washes, \$8,685 collected | Ready — the demo lists them |### What it cannot answer- **Anything causal.** There is no experiment in this file. Every effect above is observational, and  the two biggest (dormancy, price step) both have live alternative explanations — reverse causation  for the first, collinearity with month 1 for the second. An A/B holdout is the only fix.- **Retail (non-member) behaviour.** The export is members only, so it cannot speak to  member-vs-retail mix, conversion, or the cannibalisation questions the main proforma model works on.- **Seasonality.** 11 months of a ramping book. The `month` term in §6 is calendar time, not season.- **Site comparison.** Site 3 churns at 38.9% vs site 2 at 25.1%, but site 3 is 108 members with a  different cohort mix — the difference is not attributable to the site on this evidence.- **Demographics or trade area.** No geography beyond a licence-plate state that is 82.7% Kentucky.  Consistent with the panel-wide finding that trade-area measures do not predict wash volume.### Where it plugs into the rest of the repoThe main proforma model forecasts **wash counts and membership share** for a site. This file is thefirst one in the repo that describes **individual member behaviour underneath that share** — the30-day renewal hazard, the promo cliff, and the multi-car effect are all inputs the site-level modelcurrently assumes away. The natural next step is to check whether the ~7.5% monthly churn and themonth-1 cliff hold across the ~2,100-site panel, and if so, to let the membership ramp in`coldstart` inherit a real hazard curve instead of a fitted shape.

In [ ]:
live = P.score_live_book(panel, mod, cust)
dormant = P.dormant_payers(cust, 45)
print(f"active members            {len(live)}")
print(f"expected churn next cycle {live.churn_risk.sum():.1f} members ({live.churn_risk.mean():.1%})")
print(f"MRR at risk               ${live.monthly_revenue_at_risk.sum():,.0f} of ${live.arpu.sum():,.0f}")
print(f"dormant payers (45d+)     {len(dormant)} members = ${dormant.arpu.sum():,.0f}/mo, "
      f"{len(dormant)/len(live):.1%} of the active book")
live.head(12)[["customer_id", "package", "arpu", "washes_per_month", "tenure_months",
               "days_since_wash", "n_vehicles", "churn_risk", "monthly_revenue_at_risk"]].round(3)

**Insights**- **Reading — the book expects to lose ~22.8 of its 334 active members next cycle (6.8%), worth  ~\$564 of \$9,726 MRR.** That is the number the demo puts on screen, and it is trustworthy in  aggregate because the model is calibrated (§6).- **Reading — 41 active members (12.3%) are being billed \$973/month while not having washed in  45+ days.** This is the single most actionable list in the analysis.- **So-what — risk is broad rather than concentrated**, which is the honest caveat on the whole  exercise: individual risks top out around 48%, so the top-20 list carries only ~\$61/month of  expected loss. The model is a triage tool for *campaign targeting*, not a basis for  member-by-member intervention decisions.- **Caveat — members scored off a stale last cycle drift.** A member charged 25 days ago is scored  on 25-day-old behaviour; in production this would re-score nightly.

---## Reproducing this```bashconda activate sonnysjupyter lab experiments/customer-profiling/customer_profiling.ipynb   # this notebookstreamlit run experiments/customer-profiling/app.py                   # the demo```Everything above comes from [`profiling.py`](profiling.py); the demo imports the same functions, sothe two cannot disagree. Per `CLAUDE.md`, `experiments/` is standalone and off the import path —nothing here is imported by the proforma model or the API.